In [ ]:
import re
import requests
from bs4 import BeautifulSoup
import json
from sentence_transformers import SentenceTransformer
import numpy as np
from collections import defaultdict

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://www.avito.ru/',
}

session = requests.Session()
session.cookies.set('_ga', 'dummy')
session.cookies.set('_ym_uid', 'dummy')

url = input('Ссылка на объявление: ').strip()

try:
    resp = session.get(url, headers=headers, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    # ID
    id_span = soup.find('span', {'data-marker': 'item-view/item-id'})
    if id_span:
        text = id_span.get_text(strip=True)
        match = re.search(r'(\d+)', text)
        item_id = match.group(1) if match else None
    else:
        item_id = None

    # Описание
    desc_div = soup.find('div', {'data-marker': 'item-view/item-description'})
    if desc_div:
        description = ' '.join(desc_div.stripped_strings)
    else:
        description = 'Описание не найдено'

    classified = {'itemId': item_id, 'description': description}
    print(classified)

except Exception as e:
    print(f'Ошибка: {e}')

{'itemId': '7982181694', 'description': 'дисплей работает не зависает также работает все остальное'}


In [66]:
def clean_description(description):
    # Оставляем только:
    # - буквы: a-z A-Z а-я А-Я ё Ё
    # - цифры: 0-9
    # - пробелы и стандартные знаки препинания
    pattern = r'[^a-zA-Zа-яА-ЯёЁ0-9\s\.,!?;:()\[\]{}\'"-+=/&*#@$%~`_—-]'
    cleaned = re.sub(pattern, '', description)
    # Убираем лишние пробелы (множественные заменяем на один)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

# Применяем
description_raw = classified['description']
cleaned_description = clean_description(description_raw)

print(cleaned_description)

дисплей работает не зависает также работает все остальное


In [67]:
def normalize_text(text):
    # 1. Приводим к нижнему регистру
    text = text.lower()
    
    # 2. Уменьшаем повторяющиеся буквы (3+ одинаковых подряд → 2)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Удаляем повторяющиеся знаки препинания
    text = re.sub(r'([.,!?;:"-+=/&*#@$%~`_—-]){2,}', r'\1', text)
    
    return text

# Применяем к cleaned_description
normalized_description = normalize_text(cleaned_description)

print(normalized_description)

дисплей работает не зависает также работает все остальное


In [68]:
def split_into_chunks(text):
    # делим по всем основным разделителям
    chunks = re.split(r'[.!?;,:\(\)\[\]\{\}/\—\_]|\sи\s|\sили\s', text)

    # чистим пробелы и убираем пустые чанки
    clean_chunks = []

    for c in chunks:
        c = c.strip()
        if not c:
            continue

        # удалить только цифры
        if re.fullmatch(r'\d+', c):
            continue

        # удалить одиночные бесполезные слова
        if c in {"я", "мы"}:
            continue

        clean_chunks.append(c)

    return clean_chunks

# Применяем к нормализованному тексту
chunks_of_description = split_into_chunks(normalized_description)

print(chunks_of_description)

['дисплей работает не зависает также работает все остальное']
